# 🐍 Mamba K2 Training — NeuralAI's 2nd Owned Base Model

**Base**: `state-spaces/mamba-790m-hf` (793M params, 48 layers, 1536 hidden)
**Goal**: 500–1000 steps, 10K+ SFT samples, then DPO
**Runtime**: GPU required (T4 minimum, A100 recommended)

---
## ⚡ Quick Start
1. Set runtime to **T4 GPU** (Runtime → Change runtime type)
2. Run cells in order
3. After SFT: model saved to `./k2-sft-final/`
4. After DPO: merged model at `./k2-dpo-final/`
5. Download or push to HuggingFace

In [ ]:
# ============================
# CELL 1: Install Dependencies
# ============================
!pip install -q torch transformers datasets peft trl accelerate mamba-ssm bitsandbytes wandb

import os
import json
import torch
from datetime import datetime

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# ============================
# CELL 2: Config
# ============================

BASE_MODEL = "state-spaces/mamba-790m-hf"
OUTPUT_DIR = "./k2-sft-final"
DPO_OUTPUT_DIR = "./k2-dpo-final"
ADAPTER_DIR = "./k2-sft-adapter"

SFT_CONFIG = {
    "max_steps": 500,           # 500-1000 steps
    "per_device_batch_size": 2,  # GPU-dependent; T4=2, A100=4-8
    "gradient_accumulation": 4,  # effective batch = 2*4 = 8
    "learning_rate": 2e-4,
    "warmup_ratio": 0.1,
    "lr_scheduler": "cosine",
    "weight_decay": 0.01,
    "bf16": True,
    "logging_steps": 10,
    "save_steps": 100,
    "eval_steps": 100,
    "save_total_limit": 3,
}

DPO_CONFIG = {
    "max_steps": 300,
    "per_device_batch_size": 1,
    "gradient_accumulation": 4,
    "learning_rate": 5e-5,
    "warmup_ratio": 0.1,
    "bf16": True,
    "logging_steps": 10,
    "max_length": 1024,
    "max_prompt_length": 512,
    "beta": 0.1,
}

# LoRA config (Mamba SSM layers targeted)
LORA_CONFIG = {
    "r": 32,
    "lora_alpha": 64,
    "target_modules": ["x_proj", "in_proj", "out_proj", "dt_proj"],  # Mamba-specific
    "lora_dropout": 0.05,
    "bias": "none",
}

print("Config loaded\n")
print(f"Base: {BASE_MODEL}")
print(f"SFT steps: {SFT_CONFIG['max_steps']} (effective batch: {SFT_CONFIG['per_device_batch_size'] * SFT_CONFIG['gradient_accumulation']})")
print(f"LoRA rank: {LORA_CONFIG['r']}, alpha: {LORA_CONFIG['lora_alpha']}")

In [ ]:
# ============================
# CELL 3: Load & Prepare Dataset (~10K+ samples)
# ============================

from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

CHAT_TEMPLATE = "{%- for message in messages %}{%- if message['role'] == 'system' %}<|system|>\n{{ message['content'] | trim }}\n{%- elif message['role'] == 'user' %}<|user|>\n{{ message['content'] | trim }}\n{%- elif message['role'] == 'assistant' %}<|assistant|>\n{{ message['content'] | trim }}\n{%- endif %}{%- endfor -%}{%- if add_generation_prompt %}<|assistant|>\n{%- endif -%}"

tokenizer.chat_template = CHAT_TEMPLATE
print("Chat template set: Mamba K1 format (ChatML-style)")
print(f"Vocab size: {tokenizer.vocab_size}, EOS: {tokenizer.eos_token}, PAD: {tokenizer.pad_token}")

In [ ]:
# Load multiple open datasets — target 10K-15K dialogue/instruction samples

def chatml_format(example):
    """Convert raw examples to ChatML messages list"""
    messages = [
        {"role": "system", "content": "You are Mamba K2, NeuralAI's next-generation AI assistant. You are helpful, accurate, and efficient. Answer clearly using structured formatting when appropriate."},
        {"role": "user", "content": example.get("instruction") or example.get("prompt") or example.get("question", "")},
        {"role": "assistant", "content": example.get("output") or example.get("response") or example.get("answer", "")}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

def ultrachat_format(example):
    messages = example.get("messages", [])
    if not messages:
        return {"text": ""}
    formatted = [{"role": "system", "content": "You are Mamba K2, NeuralAI's next-generation AI assistant. You are helpful, accurate, and efficient."}]
    for m in messages[-3:]:  # last 3 turns
        role = "user" if m.get("role") == "user" else "assistant"
        content = m.get("content", "")
        if role in ("user", "assistant"):
            formatted.append({"role": role, "content": content})
    text = tokenizer.apply_chat_template(formatted, tokenize=False, add_generation_prompt=False)
    return {"text": text}

# Dataset 1: OpenAssistant (6K+ high-quality conversations)
print("Loading OpenAssistant (oasst1)...")
oasst = load_dataset("OpenAssistant/oasst1", split="train")
oasst_eng = oasst.filter(lambda x: x.get("lang") == "en")
print(f"  OASST English samples: {len(oasst_eng)}")

# Dataset 2: UltraChat (200K+; sample 8K)
print("Loading UltraChat (8K sample)...")
try:
    ultra = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft[:8000]")
    print(f"  UltraChat samples: {len(ultra)}")
except:
    ultra = load_dataset("stingning/ultrachat", split="train[:8000]")
    print(f"  UltraChat fallback samples: {len(ultra)}")

# Dataset 3: Alpaca (52K instructions; sample 3K)
print("Loading Alpaca (3K sample)...")
alpaca = load_dataset("tatsu-lab/alpaca", split="train[:3000]")
print(f"  Alpaca samples: {len(alpaca)}")

print(f"\nTotal raw samples: {len(oasst_eng) + len(ultra) + len(alpaca)}")

In [ ]:
# Format and combine

def prepare_oasst(examples):
    texts = []
    for i in range(len(examples.get('text', []))):
        role = examples['role'][i] if 'role' in examples else 'user'
        content = examples['text'][i] if 'text' in examples else ''
        if role == 'prompter':
            role = 'user'
        elif role == 'assistant':
            role = 'assistant'
        else:
            continue
        messages = [
            {"role": "system", "content": "You are Mamba K2, NeuralAI's next-generation AI assistant."},
            {"role": role, "content": content}
        ]
        texts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

# Format datasets
ultra_formatted = ultra.map(ultrachat_format, remove_columns=ultra.column_names)
alpaca_formatted = alpaca.map(chatml_format, remove_columns=alpaca.column_names)

# OASST needs grouping by message_tree_id first - simplified: take first 2000
oasst_sample = oasst_eng.select(range(min(2000, len(oasst_eng))))
# Convert oasst rows to a simple text format
def oasst_simple(examples):
    texts = []
    for i in range(len(examples.get('text', []))):
        role = 'assistant' if 'assistant' in str(examples.get('role', [''])[i]) else 'user'
        content = examples['text'][i]
        if not content or len(content.strip()) < 10:
            continue
        messages = [
            {"role": "system", "content": "You are Mamba K2, NeuralAI's next-generation AI assistant."},
            {"role": role, "content": content[:500]}
        ]
        texts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

oasst_formatted = oasst_sample.map(oasst_simple, batched=True, remove_columns=oasst_sample.column_names)

# Combine
from datasets import concatenate_datasets
combined = concatenate_datasets([ultra_formatted, alpaca_formatted, oasst_formatted])
combined = combined.filter(lambda x: len(x['text']) > 50)
print(f"Total training samples after formatting: {len(combined)}")
print(f"Sample text length (avg): {sum(len(t) for t in combined['text'][:100]) / min(100, len(combined)):.0f} chars")
print(f"\nSample:\n{combined[0]['text'][:400]}...")

In [ ]:
# ============================
# CELL 6: SFT Training
# ============================

from transformers import TrainingArguments
from trl import SFTTrainer
from peft import LoraConfig, get_peft_model, TaskType

print("Loading base model...")

from transformers import MambaForCausalLM
model = MambaForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    **LORA_CONFIG
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=SFT_CONFIG["per_device_batch_size"],
    gradient_accumulation_steps=SFT_CONFIG["gradient_accumulation"],
    max_steps=SFT_CONFIG["max_steps"],
    learning_rate=SFT_CONFIG["learning_rate"],
    warmup_ratio=SFT_CONFIG["warmup_ratio"],
    lr_scheduler_type=SFT_CONFIG["lr_scheduler"],
    weight_decay=SFT_CONFIG["weight_decay"],
    bf16=SFT_CONFIG["bf16"],
    logging_steps=SFT_CONFIG["logging_steps"],
    save_steps=SFT_CONFIG["save_steps"],
    save_total_limit=SFT_CONFIG["save_total_limit"],
    report_to="wandb" if os.environ.get("WANDB_API_KEY") else "none",
    run_name=f"mamba-k2-sft-{datetime.now().strftime('%Y%m%d-%H%M')}",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=combined,
    max_seq_length=1024,
    dataset_text_field="text",
)

print(f"Starting SFT training — {SFT_CONFIG['max_steps']} steps...")
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nSFT complete! Model saved to {OUTPUT_DIR}")

In [ ]:
# ============================
# CELL 7: Quick Generation Test
# ============================

test_prompts = [
    "Explain quantum entanglement in 2 sentences.",
    "Write a Python function to reverse a linked list.",
    "What are the three branches of the US government and their roles?",
    "Give me a table comparing Python, JavaScript, and Rust for web development.",
]

model.eval()
for prompt in test_prompts:
    messages = [
        {"role": "system", "content": "You are Mamba K2, NeuralAI's next-generation AI assistant. Be helpful, accurate, and use structured formatting."},
        {"role": "user", "content": prompt}
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt", add_generation_prompt=True).to(model.device)
    outputs = model.generate(
        inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"\n{'='*60}")
    print(f"Q: {prompt}")
    print(f"A: {response[:300]}")

---
## DPO Training (Optional — requires preference pairs)
---

In [ ]:
# ============================
# CELL 8: DPO Training
# ============================
# Requires a preference dataset (chosen/rejected pairs)
# Example: Anthropic HH-RLHF or custom DPO dataset

RUN_DPO = False  # Set to True when you have DPO data

if RUN_DPO:
    from trl import DPOTrainer
    
    # Load DPO dataset (example: Anthropic HH-RLHF)
    dpo_data = load_dataset("Anthropic/hh-rlhf", split="train[:5000]")
    
    def format_dpo(example):
        return {
            "prompt": f"<|system|>\nYou are Mamba K2, NeuralAI's next-generation AI assistant.\n<|user|>\n{example['chosen'].split('Assistant:')[0].replace('Human:', '').strip()}\n<|assistant|>\n",
            "chosen": example['chosen'].split('Assistant:')[-1].strip(),
            "rejected": example['rejected'].split('Assistant:')[-1].strip(),
        }
    
    dpo_data = dpo_data.map(format_dpo)
    
    dpo_trainer = DPOTrainer(
        model=model,
        tokenizer=tokenizer,
        args=TrainingArguments(
            output_dir=DPO_OUTPUT_DIR,
            per_device_train_batch_size=DPO_CONFIG["per_device_batch_size"],
            gradient_accumulation_steps=DPO_CONFIG["gradient_accumulation"],
            max_steps=DPO_CONFIG["max_steps"],
            learning_rate=DPO_CONFIG["learning_rate"],
            warmup_ratio=DPO_CONFIG["warmup_ratio"],
            bf16=DPO_CONFIG["bf16"],
            logging_steps=DPO_CONFIG["logging_steps"],
            report_to="wandb" if os.environ.get("WANDB_API_KEY") else "none",
            run_name=f"mamba-k2-dpo-{datetime.now().strftime('%Y%m%d-%H%M')}",
        ),
        max_length=DPO_CONFIG["max_length"],
        max_prompt_length=DPO_CONFIG["max_prompt_length"],
        beta=DPO_CONFIG["beta"],
        train_dataset=dpo_data,
    )
    
    print(f"Starting DPO training — {DPO_CONFIG['max_steps']} steps...")
    dpo_trainer.train()
    dpo_trainer.save_model(DPO_OUTPUT_DIR)
    print(f"\nDPO complete! Model saved to {DPO_OUTPUT_DIR}")

else:
    print("DPO skipped. Set RUN_DPO=True with a preference dataset to train.")

In [ ]:
# ============================
# CELL 9: Merge & Push to HuggingFace
# ============================

from peft import PeftModel

# Merge adapter into base
print("Merging LoRA adapter into base model...")
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./k2-merged")
tokenizer.save_pretrained("./k2-merged")
print("Merged model saved to ./k2-merged/")

# Push to HuggingFace (uncomment when ready)
# from huggingface_hub import HfApi
# api = HfApi()
# api.create_repo("Subject-Emu-5259/Mamba-K2", private=True, exist_ok=True)
# merged_model.push_to_hub("Subject-Emu-5259/Mamba-K2")
# tokenizer.push_to_hub("Subject-Emu-5259/Mamba-K2")
# print("Pushed to https://huggingface.co/Subject-Emu-5259/Mamba-K2")